# Analysis Output Review

This notebook reviews the production analysis artifacts written by `run_factor_research`.

It is intentionally notebook-facing only: it reads the persisted V2 outputs rather than reimplementing the pipeline.

## Setup

Adjust `ANALYSIS_DIR` if your outputs are not under the default `data/analysis` directory.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    import seaborn as sns
except ImportError:
    sns = None

In [ ]:
cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
ANALYSIS_DIR = repo_root / "data" / "analysis"

OUTPUT_STEMS = {
    "panel": "analysis_snapshot_panel",
    "factor_returns": "analysis_factor_returns",
    "factor_registry": "analysis_factor_registry",
    "candidate_diagnostics": "analysis_factor_candidate_diagnostics",
    "distinctness": "analysis_factor_distinctness",
    "selection_scores": "analysis_factor_selection_scores",
    "screening_decisions": "analysis_factor_screening_decisions",
    "cluster_membership": "analysis_factor_cluster_membership",
    "factor_diagnostics": "analysis_factor_diagnostics",
    "factor_model_telemetry": "analysis_factor_model_telemetry",
    "factor_final_vif_diagnostics": "analysis_factor_final_vif_diagnostics",
    "factor_persistence": "analysis_factor_persistence",
    "model_results": "analysis_model_results",
    "current_betas": "analysis_current_betas",
    "asset_expected_returns": "analysis_asset_expected_returns",
    "asset_factor_betas": "analysis_asset_factor_betas",
}


def read_output(stem: str) -> pd.DataFrame:
    path = ANALYSIS_DIR / f"{stem}.parquet"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path)


artifacts = {name: read_output(stem) for name, stem in OUTPUT_STEMS.items()}
summary = pd.DataFrame(
    [
        {
            "artifact": name,
            "rows": int(df.shape[0]),
            "columns": int(df.shape[1]),
            "path": str(ANALYSIS_DIR / f"{OUTPUT_STEMS[name]}.parquet"),
        }
        for name, df in artifacts.items()
    ]
).sort_values("artifact")

display(summary)

In [ ]:
panel = artifacts["panel"]
factor_returns = artifacts["factor_returns"]
factor_registry = artifacts["factor_registry"]
candidate_diagnostics = artifacts["candidate_diagnostics"]
distinctness = artifacts["distinctness"]
selection_scores = artifacts["selection_scores"]
screening_decisions = artifacts["screening_decisions"]
cluster_membership = artifacts["cluster_membership"]
factor_diagnostics = artifacts["factor_diagnostics"]
factor_model_telemetry = artifacts["factor_model_telemetry"]
factor_final_vif_diagnostics = artifacts["factor_final_vif_diagnostics"]
factor_persistence = artifacts["factor_persistence"]
model_results = artifacts["model_results"]

## Stage Summaries

In [ ]:
if not factor_registry.empty:
    display(
        factor_registry.groupby(["sleeve", "family", "kind", "admission_status"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["sleeve", "count", "family", "kind"], ascending=[True, False, True, True])
    )

if not candidate_diagnostics.empty:
    display(
        candidate_diagnostics[
            [
                "sleeve",
                "factor_id",
                "admitted_for_construction",
                "pre_compression_candidate_count",
                "post_compression_candidate_count",
                "compression_removed_count",
                "compression_removed_ratio",
            ]
        ].sort_values(["sleeve", "factor_id"])
    )

if not screening_decisions.empty:
    display(
        screening_decisions.groupby(["stage", "decision", "reason"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["stage", "count", "reason"], ascending=[True, False, True])
    )

## Final Selection And VIF Review

In [ ]:
if not factor_final_vif_diagnostics.empty:
    display(
        factor_final_vif_diagnostics.sort_values(["sleeve", "vif_rank_within_sleeve", "factor_id"])
    )
else:
    selected_mask = (
        factor_diagnostics["selected_for_model"].fillna(False)
        if "selected_for_model" in factor_diagnostics
        else pd.Series(False, index=factor_diagnostics.index)
    )
    display(factor_diagnostics.loc[selected_mask].sort_values(["sleeve", "factor_id"]))

if not factor_model_telemetry.empty:
    display(
        factor_model_telemetry[
            [
                "sleeve",
                "factor_id",
                "selection_count",
                "selection_frequency",
                "sign_consistency",
                "mean_abs_beta",
                "mean_selected_r2_test",
                "is_persistent",
                "final_selected",
                "final_vif",
            ]
        ].sort_values(["sleeve", "final_selected", "selection_frequency", "factor_id"], ascending=[True, False, False, True])
    )

In [ ]:
if {"trade_date", "factor_id", "factor_return"}.issubset(factor_returns.columns):
    factor_returns_wide = factor_returns.pivot(
        index="trade_date",
        columns="factor_id",
        values="factor_return",
    ).sort_index()
else:
    factor_returns_wide = factor_returns.copy()
    if "trade_date" in factor_returns_wide.columns:
        factor_returns_wide = factor_returns_wide.set_index("trade_date")
    elif "index" in factor_returns_wide.columns:
        factor_returns_wide = factor_returns_wide.rename(columns={"index": "trade_date"}).set_index("trade_date")
    elif factor_returns_wide.index.name != "trade_date":
        factor_returns_wide.index.name = "trade_date"
    factor_returns_wide = factor_returns_wide.sort_index()


def final_factor_ids(sleeve: str | None = None) -> list[str]:
    if not factor_final_vif_diagnostics.empty:
        df = factor_final_vif_diagnostics
    else:
        if factor_diagnostics.empty:
            return []
        mask = (
            factor_diagnostics["selected_for_model"].fillna(False)
            if "selected_for_model" in factor_diagnostics
            else pd.Series(False, index=factor_diagnostics.index)
        )
        df = factor_diagnostics.loc[mask, ["factor_id", "sleeve"]]
    if sleeve is not None:
        df = df.loc[df["sleeve"] == sleeve]
    return sorted(df["factor_id"].dropna().astype(str).unique().tolist())


def final_factor_corr(sleeve: str) -> pd.DataFrame:
    factor_ids = final_factor_ids(sleeve)
    if not factor_ids:
        return pd.DataFrame()
    return factor_returns_wide.reindex(columns=factor_ids).corr()


def plot_corr(corr: pd.DataFrame, title: str) -> None:
    if corr.empty:
        print(f"No final selected factors for {title}.")
        return
    display(corr.round(3))
    if plt is None or sns is None:
        return
    plt.figure(figsize=(max(6, len(corr.columns) * 0.8), max(4, len(corr.columns) * 0.6)))
    sns.heatmap(corr, cmap="RdBu_r", center=0.0, vmin=-1.0, vmax=1.0, annot=True, fmt=".2f")
    plt.title(title)
    plt.tight_layout()
    plt.show()


sleeves = sorted(panel.get("sleeve", pd.Series(dtype=str)).dropna().astype(str).unique().tolist())
for sleeve in sleeves:
    plot_corr(final_factor_corr(sleeve), f"Final factor return correlation: {sleeve}")

## Suspicious Pair Review

Use this section to inspect the V1-style comparisons directly from stored factor returns.

In [ ]:
all_factor_ids = sorted(factor_returns_wide.columns.astype(str).tolist()) if not factor_returns_wide.empty else []


def matching_factor_ids(fragment: str) -> list[str]:
    fragment = fragment.lower()
    return [factor_id for factor_id in all_factor_ids if fragment in factor_id.lower()]


def compare_factor_pair(left_factor_id: str, right_factor_id: str) -> pd.DataFrame:
    pair = factor_returns_wide.reindex(columns=[left_factor_id, right_factor_id]).dropna()
    if pair.empty:
        return pd.DataFrame(
            [{"left_factor_id": left_factor_id, "right_factor_id": right_factor_id, "abs_corr": np.nan, "rows": 0}]
        )
    corr = pair.corr().iloc[0, 1]
    if plt is not None:
        pair.plot(title=f"{left_factor_id} vs {right_factor_id}", figsize=(10, 4))
        plt.tight_layout()
        plt.show()
    return pd.DataFrame(
        [{
            "left_factor_id": left_factor_id,
            "right_factor_id": right_factor_id,
            "abs_corr": float(abs(corr)),
            "rows": int(len(pair)),
        }]
    )


comparison_fragments = [
    ("currency__usd", "country__usa"),
    ("raw__country__", "grouped__continent__"),
    ("raw__country__", "grouped__bloc__"),
    ("raw__industry__", "grouped__supersector__"),
    ("macro__", "macro_bloc__"),
]

for left_fragment, right_fragment in comparison_fragments:
    print({
        "left_fragment": left_fragment,
        "left_matches": matching_factor_ids(left_fragment)[:10],
        "right_fragment": right_fragment,
        "right_matches": matching_factor_ids(right_fragment)[:10],
    })

## Distinctness And Cluster Review

In [ ]:
if not distinctness.empty:
    display(distinctness.sort_values(["comparison_stage", "abs_correlation"], ascending=[True, False]).head(50))

if not cluster_membership.empty:
    display(cluster_membership.sort_values(["sleeve", "cluster_id", "keep_factor", "factor_id"], ascending=[True, True, False, True]))

if not selection_scores.empty:
    display(selection_scores.sort_values(["stage", "selection_score", "factor_id"], ascending=[True, False, True]).head(100))